<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/Wikipedia_Knowledge_Universe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wikipedia Knowledge Universe
Explore human knowledge as a navigable, interactive graph. This notebook builds a visual map of Wikipedia topics, analyzing their relationships and importance using network science and AI.

In [1]:
!pip install -q pyvis networkx plotly pandas community sentence-transformers wikipedia-api tqdm ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.9 MB/s eta 0:00:00


In [2]:
import networkx as nx
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pyvis.network import Network
import wikipediaapi
from sentence_transformers import SentenceTransformer
from community import community_louvain
import ipywidgets as widgets
from IPython.display import display, HTML
from tqdm.auto import tqdm
import json
import requests

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Data Acquisition & Graph Construction
We use the Wikipedia API to fetch a set of core topics and their related pages to form our initial knowledge graph.

In [3]:
def fetch_wikipedia_knowledge_graph(seed_topics, max_nodes=500):
    """Fetches Wikipedia pages and links to build a NetworkX graph."""
    wiki = wikipediaapi.Wikipedia('WikipediaKnowledgeUniverse/1.0 (contact: user@example.com)', 'en')
    G = nx.Graph()
    nodes_to_process = list(seed_topics)
    processed_nodes = set()

    pbar = tqdm(total=max_nodes, desc="Building Knowledge Graph")

    while nodes_to_process and len(G.nodes) < max_nodes:
        topic = nodes_to_process.pop(0)
        if topic in processed_nodes: continue

        page = wiki.page(topic)
        if not page.exists(): continue

        processed_nodes.add(topic)
        G.add_node(topic, title=page.title, url=page.fullurl, summary=page.summary[:200] + "...")

        # Add links (edges)
        links = list(page.links.keys())
        for link in links[:15]: # Limit links per page for performance
            if len(G.nodes) >= max_nodes: break
            G.add_edge(topic, link)
            if link not in processed_nodes:
                nodes_to_process.append(link)

        pbar.update(1)

    pbar.close()
    return G

# Define starting points for the universe
seeds = ["Science", "Technology", "History", "Art", "Philosophy", "Mathematics", "Biology"]
G = fetch_wikipedia_knowledge_graph(seeds, max_nodes=1000)

print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Building Knowledge Graph:   0%|          | 0/1000 [00:00<?, ?it/s]

Graph built with 1000 nodes and 1333 edges.


## 2. Graph Analytics & Community Detection
We calculate the importance of each node and group them into logical communities using the Louvain algorithm.

In [4]:
# Calculate Metrics
print("Calculating network metrics...")
pagerank = nx.pagerank(G)
betweenness = nx.betweenness_centrality(G)
degree = dict(G.degree())

# Community Detection
partition = community_louvain.best_partition(G)

# Add attributes to nodes
for node in G.nodes():
    G.nodes[node]['pagerank'] = pagerank[node]
    G.nodes[node]['betweenness'] = betweenness[node]
    G.nodes[node]['degree'] = degree[node]
    G.nodes[node]['community'] = partition[node]
    G.nodes[node]['size'] = pagerank[node] * 5000  # Scaling for visualization

# Summary Statistics Table
stats_df = pd.DataFrame({
    'Node': list(G.nodes()),
    'PageRank': [pagerank[n] for n in G.nodes()],
    'Community': [partition[n] for n in G.nodes()],
    'Degree': [degree[n] for n in G.nodes()]
}).sort_values(by='PageRank', ascending=False)

print("Top 10 Most Influential Topics (PageRank):")
display(stats_df.head(10))

Calculating network metrics...
Top 10 Most Influential Topics (PageRank):


,Node,PageRank,Community,Degree
33,1948 Palestine war,0.007565,16,18
48,A Room of One's Own,0.007429,20,25
83,Addison-Wesley Publishing Company,0.007296,8,16
76,A Mathematician's Apology,0.007296,25,16
1,19th century in science,0.007225,1,16
4,A. Rupert Hall,0.007225,4,16
17,3D printing,0.007215,11,16
64,Abortion,0.007139,24,17
86,Adrien-Marie Legendre,0.007077,6,16
47,291 (art gallery),0.007064,21,16


## 3. Statistics Dashboard
A high-level overview of the Knowledge Universe structure.

In [5]:
def show_dashboard(G):
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    avg_degree = sum(dict(G.degree()).values()) / num_nodes
    density = nx.density(G)

    fig = go.Figure()

    metrics = [
        ("Nodes", num_nodes),
        ("Edges", num_edges),
        ("Avg Degree", round(avg_degree, 2)),
        ("Density", f"{density:.4f}")
    ]

    for i, (label, val) in enumerate(metrics):
        fig.add_trace(go.Indicator(
            mode = "number+delta",
            value = val if isinstance(val, (int, float)) else 0,
            title = {"text": label},
            domain = {'x': [i/4, (i+1)/4], 'y': [0, 1]}
        ))

    fig.update_layout(height=300, title="Universe Vital Signs")
    fig.show()

show_dashboard(G)

## 4. Visualizations: 2D & 3D Knowledge Maps
Explore the network interactively. The 2D view uses physics for discovery, while the 3D view provides a 'Universe' perspective.

In [10]:
def create_pyvis_graph(G, filename='knowledge_graph.html'):
    net = Network(height='700px', width='100%', bgcolor='#222222', font_color='white', notebook=True, cdn_resources='remote')

    # Map communities to colors
    colors = px.colors.qualitative.Plotly

    for node, attrs in G.nodes(data=True):
        # Fix: use .get() to handle missing attributes like 'summary' or 'community'
        comm = attrs.get('community', 0)
        summary = attrs.get('summary', 'No summary available.')
        size = attrs.get('size', 10)

        color = colors[comm % len(colors)]
        net.add_node(node, label=node, title=f"{node}<br>{summary}",
                     color=color, size=size)

    for source, target in G.edges():
        net.add_edge(source, target, color='#555555')

    net.set_options("""
    var options = {
      "physics": {
        "forceAtlas2Based": {"gravitationalConstant": -50, "centralGravity": 0.01, "springLength": 100, "springConstant": 0.08},
        "maxVelocity": 50, "solver": "forceAtlas2Based", "timestep": 0.35, "stabilization": {"iterations": 150}
      }
    }
    """)
    return net.show(filename)

# Render 2D Graph
create_pyvis_graph(G)

knowledge_graph.html


In [7]:
def create_3d_universe(G):
    pos = nx.spring_layout(G, dim=3, seed=42)

    edge_x, edge_y, edge_z = [], [], []
    for edge in G.edges():
        x0, y0, z0 = pos[edge[0]]
        x1, y1, z1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        edge_z.extend([z0, z1, None])

    edge_trace = go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, line=dict(width=1, color='#888'), hoverinfo='none', mode='lines')

    node_x, node_y, node_z = [], [], []
    node_text, node_color, node_size = [], [], []

    colors = px.colors.qualitative.Plotly

    for node in G.nodes():
        x, y, z = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_z.append(z)
        node_text.append(f"{node} (Community {G.nodes[node]['community']})")
        node_color.append(colors[G.nodes[node]['community'] % len(colors)])
        node_size.append(G.nodes[node]['pagerank'] * 1000 + 5)

    node_trace = go.Scatter3d(
        x=node_x, y=node_y, z=node_z, mode='markers',
        hoverinfo='text', text=node_text,
        marker=dict(showscale=False, color=node_color, size=node_size, line_width=2, opacity=0.8)
    )

    fig = go.Figure(data=[edge_trace, node_trace],
                 layout=go.Layout(
                    title='3D Knowledge Universe',
                    template='plotly_dark',
                    scene=dict(xaxis=dict(showbackground=False), yaxis=dict(showbackground=False), zaxis=dict(showbackground=False)),
                    margin=dict(b=0, l=0, r=0, t=40)
                 ))
    fig.show()

create_3d_universe(G)

## 5. AI Features: Semantic Search & Path Finder
Use AI to search by meaning and calculate the shortest path between distant concepts.

In [8]:
print("Loading AI model for semantic search...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Precompute embeddings for all nodes
nodes = list(G.nodes())
node_embeddings = model.encode(nodes, show_progress_bar=True)

def semantic_search(query, top_k=5):
    query_embedding = model.encode([query])
    similarities = np.dot(node_embeddings, query_embedding.T).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(nodes[i], similarities[i]) for i in top_indices]

def find_knowledge_path(source, target):
    try:
        path = nx.shortest_path(G, source=source, target=target)
        return path
    except nx.NetworkXNoPath:
        return None

# Example Search
results = semantic_search("Artificial Intelligence")
print("Semantic Search Results:", results)

Loading AI model for semantic search...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Semantic Search Results: [('Artificial intelligence', np.float32(1.0)), ('AI', np.float32(0.7912463)), ('Artificial general intelligence', np.float32(0.7503011)), ('Artificial Intelligence Act', np.float32(0.70672834)), ('AI agent', np.float32(0.6805278))]


## 6. Hierarchical Views & Exports
Drill down into the data with Sunburst charts and export your graph for external use.

## 7. Interactive Knowledge Explorer
Use the widgets below to find paths between topics or perform semantic searches across the universe.

In [11]:
# UI for Path Finder
node_list = sorted(list(G.nodes()))
src_dropdown = widgets.Dropdown(options=node_list, value=node_list[0], description='Source:')
tgt_dropdown = widgets.Dropdown(options=node_list, value=node_list[-1], description='Target:')
path_btn = widgets.Button(description="Find Path", button_style='info')
output = widgets.Output()

def on_path_click(b):
    with output:
        output.clear_output()
        path = find_knowledge_path(src_dropdown.value, tgt_dropdown.value)
        if path:
            print(f"Shortest Path Found: {' -> '.join(path)}")
        else:
            print("No path exists between these topics in the current sample.")

path_btn.on_click(on_path_click)
display(widgets.VBox([widgets.HBox([src_dropdown, tgt_dropdown]), path_btn, output]))

# UI for Semantic Search
search_input = widgets.Text(placeholder='Type a concept...', description='AI Search:')
search_btn = widgets.Button(description="Search", button_style='success')
search_output = widgets.Output()

def on_search_click(b):
    with search_output:
        search_output.clear_output()
        results = semantic_search(search_input.value)
        print(f"Top AI Matches for '{search_input.value}':")
        for topic, score in results:
            print(f"- {topic} (Similarity: {score:.2f})")

search_btn.on_click(on_search_click)
display(widgets.VBox([search_input, search_btn, search_output]))

In [9]:
def create_sunburst(G):
    data = []
    for node, attrs in G.nodes(data=True):
        data.append({
            'Topic': node,
            'Community': f"Cluster {attrs['community']}",
            'Importance': attrs['pagerank']
        })
    df = pd.DataFrame(data)
    fig = px.sunburst(df, path=['Community', 'Topic'], values='Importance',
                      title="Knowledge Hierarchy by Community", template='plotly_dark')
    fig.show()

def export_data(G):
    # Save GraphML and JSON
    nx.write_graphml(G, "knowledge_universe.graphml")
    with open("knowledge_universe.json", "w") as f:
        json.dump(nx.node_link_data(G), f)
    print("Graph exported to GraphML and JSON formats.")

create_sunburst(G)
export_data(G)

Graph exported to GraphML and JSON formats.
